# 02 · Feature engineering

All feature logic lives in **`src/churnpulse/features.py`** — shared by this notebook, training, the Flink streaming job, the API and the action center (no training/serving skew). Every feature belongs to one persona category, which drives the dashboard's grouped explanations.

In [1]:
%matplotlib inline
import warnings, os
warnings.filterwarnings("ignore")
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from churnpulse import config
from churnpulse.dataset import load_customers
from churnpulse.features import FEATURE_CATEGORY, CATEGORIES
sns.set_theme(style="whitegrid", palette=["#0aa38a", "#f43f5e", "#f59e0b", "#6366f1", "#38bdf8"])
CAT_COLORS = {"geographic": "#38bdf8", "demographic": "#a78bfa", "technographic": "#22d3ee",
              "behavioural": "#fbbf24", "psychographic": "#f472b6"}
pd.set_option("display.max_columns", 50)


In [2]:
from churnpulse.features import build_features, PROFILE_FEATURES, SURVEY_FEATURES
df = load_customers(); X = build_features(df)
pd.Series(FEATURE_CATEGORY).value_counts().rename('features').to_frame()

,features
behavioural,16
technographic,15
demographic,6
geographic,3
psychographic,3


## Engineered features
| Feature | Category | Why |
|---|---|---|
| `log_population` | geographic | area density on a sensible scale |
| `under_30`, `senior` | demographic | life-stage effects in age |
| `num_addons`, `num_protection` | technographic | more services = more locked in |
| `gb_per_dollar` | technographic | perceived value of the plan |
| `charge_increase_ratio` | behavioural | today's bill vs historic average — price rises trigger churn |
| `contract_months`, `auto_pay`, `had_refund`, `extra_data_share`, `tenure_group` | behavioural | commitment, friction, problems |
| `referred_a_friend` | psychographic | advocacy / loyalty |

In [3]:
new = ['under_30', 'senior', 'num_addons', 'num_protection', 'gb_per_dollar', 'charge_increase_ratio',
       'contract_months', 'auto_pay', 'had_refund', 'referred_a_friend']
X[new].assign(churned=df.churned).groupby('churned').mean().T.round(3)

churned,0,1
under_30,0.212,0.163
senior,0.129,0.255
num_addons,3.102,2.961
num_protection,1.399,0.895
gb_per_dollar,0.292,0.322
charge_increase_ratio,1.002,1.003
contract_months,12.100,2.568
auto_pay,0.953,0.924
had_refund,0.081,0.057
referred_a_friend,0.502,0.334


## Same function for one live customer
This is exactly what the Flink job and the API call:

In [4]:
from churnpulse.features import profile_to_frame, PROFILE_FIELDS
profile = df.iloc[0][PROFILE_FIELDS].to_dict()
profile_to_frame(profile)[SURVEY_FEATURES].T.head(15)

,0
latitude,34.02381
longitude,-118.156582
log_population,11.137534
age,78
gender,Male
married,0
num_dependents,0
under_30,0
senior,1
internet_type,DSL


## Two feature sets
* **Profile model:** every feature except the satisfaction survey.
* **Survey model:** the same plus `satisfaction_score`.

In [5]:
print(len(PROFILE_FEATURES), 'profile features;', len(SURVEY_FEATURES), 'survey features')

42 profile features; 43 survey features
